# Interior Style Classifier — Training Notebook

Trains an EfficientNet-B0 model to classify room photos into 5 interior design styles:
**Mid-Century Modern · Scandinavian · Industrial · Bohemian · Minimalist**

**Runtime:** GPU (T4 or better recommended)  
**Expected training time:** ~15 min on Colab free tier  
**Expected val accuracy:** ~80–85% on the 150-image starter dataset

---
### Two-phase transfer learning
1. **Phase 1** — Freeze the EfficientNet backbone, train only the custom classification head
2. **Phase 2** — Unfreeze the top 3 backbone blocks and fine-tune at a lower learning rate

In [ ]:
# ── 1. Clone repo and install dependencies ─────────────────────────────────
!git clone https://github.com/yourusername/interior-style-nn.git
%cd interior-style-nn
!pip install -q timm langfuse python-dotenv

In [ ]:
# ── 2. Verify GPU ──────────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Download dataset ────────────────────────────────────────────────────
# Downloads ~150 images (30 per class) from Unsplash
# Takes ~3-5 minutes depending on connection speed
!python data/scripts/download_dataset.py --per-class 30

# Verify download
import os
for style in os.listdir('data/images'):
    n = len(os.listdir(f'data/images/{style}'))
    print(f'  {style}: {n} images')

In [ ]:
# ── 4. Visualize sample images ─────────────────────────────────────────────
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

styles = sorted(os.listdir('data/images'))
fig, axes = plt.subplots(1, len(styles), figsize=(18, 4))

for ax, style in zip(axes, styles):
    style_dir = Path('data/images') / style
    imgs = list(style_dir.glob('*.jpg'))
    img = Image.open(random.choice(imgs))
    ax.imshow(img)
    ax.set_title(style.replace('_', ' ').title(), fontsize=10)
    ax.axis('off')

plt.suptitle('Sample images per class', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Build dataloaders ───────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')

from model.dataset import build_dataloaders

train_loader, val_loader, class_names = build_dataloaders(
    data_dir='data/images',
    val_split=0.2,
    batch_size=32,
    num_workers=2,
)

print(f'\nClass → index mapping:')
for i, cls in enumerate(class_names):
    print(f'  {i}: {cls}')

In [ ]:
# ── 6. Inspect a training batch ────────────────────────────────────────────
import torchvision
import numpy as np

imgs, labels = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}')  # [B, 3, 224, 224]
print(f'Labels: {labels[:8].tolist()}')

# Unnormalize for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
display = imgs[:8] * std + mean
display = display.clamp(0, 1)

grid = torchvision.utils.make_grid(display, nrow=4)
plt.figure(figsize=(12, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.title('Augmented training batch')
plt.axis('off')
plt.show()

In [ ]:
# ── 7. Initialize model ────────────────────────────────────────────────────
from model.model import InteriorStyleClassifier

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = InteriorStyleClassifier(pretrained=True).to(device)

print(f'Total parameters:     {model.total_params():,}')
print(f'Trainable parameters: {model.trainable_params():,}')
print(f'\nBackbone: EfficientNet-B0 (ImageNet pretrained)')
print(f'Head: Linear(1280→512) → BN → ReLU → Dropout → Linear(512→{len(class_names)})')

In [ ]:
# ── 8. Training loop ───────────────────────────────────────────────────────
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Config
FREEZE_EPOCHS = 5
TOTAL_EPOCHS  = 15
LR            = 1e-3
WEIGHTS_PATH  = 'weights/best_model.pth'

os.makedirs('weights', exist_ok=True)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

# ── Phase 1: frozen backbone ──────────────────────────────────────────────
print(f'=== Phase 1: Training head only (epochs 1–{FREEZE_EPOCHS}) ===')
model.freeze_backbone()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=FREEZE_EPOCHS)

for epoch in range(1, FREEZE_EPOCHS + 1):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = val_epoch(model, val_loader, criterion)
    scheduler.step()
    elapsed = time.time() - t0
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'  Epoch {epoch:02d}/{FREEZE_EPOCHS} | train_loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} acc={va:.3f} | {elapsed:.1f}s')
    if va > best_val_acc:
        best_val_acc = va
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': va, 'class_names': class_names}, WEIGHTS_PATH)
        print(f'  ✓ Saved (val_acc={va:.3f})')

# ── Phase 2: fine-tune top layers ─────────────────────────────────────────
print(f'\n=== Phase 2: Fine-tuning top blocks (epochs {FREEZE_EPOCHS+1}–{TOTAL_EPOCHS}) ===')
model.unfreeze_top_layers(num_blocks=3)
print(f'Trainable params: {model.trainable_params():,}')
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR*0.1, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - FREEZE_EPOCHS)

for epoch in range(FREEZE_EPOCHS + 1, TOTAL_EPOCHS + 1):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = val_epoch(model, val_loader, criterion)
    scheduler.step()
    elapsed = time.time() - t0
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'  Epoch {epoch:02d}/{TOTAL_EPOCHS} | train_loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} acc={va:.3f} | {elapsed:.1f}s')
    if va > best_val_acc:
        best_val_acc = va
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': va, 'class_names': class_names}, WEIGHTS_PATH)
        print(f'  ✓ Saved (val_acc={va:.3f})')

print(f'\n✅ Training complete. Best val accuracy: {best_val_acc:.3f}')

In [ ]:
# ── 9. Plot training curves ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history['train_loss']) + 1)
phase_boundary = FREEZE_EPOCHS

ax1.plot(epochs, history['train_loss'], label='train')
ax1.plot(epochs, history['val_loss'],   label='val')
ax1.axvline(phase_boundary + 0.5, color='grey', linestyle='--', alpha=0.6, label='phase 2 start')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Cross-Entropy Loss'); ax1.legend()

ax2.plot(epochs, history['train_acc'], label='train')
ax2.plot(epochs, history['val_acc'],   label='val')
ax2.axvline(phase_boundary + 0.5, color='grey', linestyle='--', alpha=0.6, label='phase 2 start')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy'); ax2.legend()

plt.suptitle('Training Curves — Interior Style Classifier', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10. Evaluate on validation set ────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Load best checkpoint
from model.model import load_model
best_model = load_model(WEIGHTS_PATH, device=device)

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = best_model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11. Inference on a single image ────────────────────────────────────────
from api.predictor import Predictor
import random

predictor = Predictor(weights_path=WEIGHTS_PATH, device=device)

# Pick a random validation image
all_styles = sorted(os.listdir('data/images'))
test_style = random.choice(all_styles)
test_imgs  = list(Path(f'data/images/{test_style}').glob('*.jpg'))
test_img   = random.choice(test_imgs)

with open(test_img, 'rb') as f:
    image_bytes = f.read()

style, confidence, all_scores, ms = predictor.predict(image_bytes)
palette = predictor.get_palette(style)

print(f'Image:      {test_img.name}')
print(f'True label: {test_style.replace("_", " ").title()}')
print(f'Predicted:  {style} ({confidence:.1%} confidence)')
print(f'Latency:    {ms:.1f}ms')
print(f'\nAll scores:')
for s, p in sorted(all_scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(p * 30)
    print(f'  {s:<22} {p:.3f}  {bar}')
print(f'\nPalette: {palette.description}')
print(f'Primary: {palette.primary}')
print(f'Accent:  {palette.accent}')

# Show the image
img = Image.open(test_img)
plt.figure(figsize=(5, 4))
plt.imshow(img)
plt.title(f'Predicted: {style} ({confidence:.1%})\nActual: {test_style.replace("_"," ").title()}')
plt.axis('off')
plt.show()

In [ ]:
# ── 12. Download weights ────────────────────────────────────────────────────
from google.colab import files
files.download(WEIGHTS_PATH)
print('weights/best_model.pth downloaded — place in your project root to use with the FastAPI server.')